In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path:/ c:\Users\Wanling Xie\FixedIncomeLib
Fixed Income Library is loaded.


### Create Build Method Collection

In [2]:
bm_list = []
# create yield curve build method
content_sofr = {
    'TARGET' : 'SOFR-1B',
    'OVERNIGHT INDEX FUTURE' : 'SOFR-FUTURE-3M',
    'OVERNIGHT INDEX SWAP' : 'USD-SOFR-OIS'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_sofr))
content_sofr_funding = {
    'TARGET' : 'SOFR-1B-FLAT',
    'SPREAD ZERO RATE' : 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_sofr_funding))
content_ff = {
    'TARGET' : 'FF-1B',
    'REFERENCE INDEX' : 'SOFR-1B',
    'OVERNIGHT INDEX BASIS SWAP' : 'USD-FF-3M-OVER-USD-SOFR-OIS-3M'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_ff))
content_ff_funding = {
    'TARGET' : 'FF-1B-FLAT',
    'SPREAD ZERO RATE' : 'FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_ff_funding))
# create yield curve common build method 
content = {
    'TARGET' : 'USD',
    'FUNDING PARAMETERS' : 'USD-FUNDING-PARAMETERS',
    'SOLVER' : 'BRENTQ'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

### Create Data Collection

In [3]:
### ois futures
df_fut = pd.DataFrame([    
    ['2026-03-19x2026-06-18', 96.44],
    ['2026-06-18x2026-09-17', 96.70],
    ['2026-09-17x2026-12-10', 96.85],
    ['2026-12-10x2027-03-17', 96.90],
    ['2027-03-17x2027-06-16', 96.91],
    ['2027-06-16x2027-09-15', 96.89],
    ['2027-09-15x2027-12-15', 96.85],
    ['2027-12-15x2028-03-15', 96.81],
    ['2028-03-15x2028-06-21', 96.76],
    ['2028-06-21x2028-09-20', 96.71],
    ['2028-09-20x2028-12-20', 96.66],
    ['2028-12-20x2029-03-21', 96.61]],
    columns=['axis1', 'values']).set_index('axis1')
data_fut = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut)

In [4]:
### ois swap
df_swap = pd.DataFrame([    
    ['4Y', 0.03358],
    ['5Y', 0.03422],
    ['6Y', 0.03491],
    ['7Y', 0.03560],
    ['8Y', 0.03624],
    ['9Y', 0.03685],
    ['10Y', 0.03742],
    ['12Y', 0.03849],
    ['15Y', 0.03974],
    ['20Y', 0.04087],
    ['25Y', 0.04110],
    ['30Y', 0.04089],
    ['35Y', 0.04044],
    ['40Y', 0.03996],
    ['50Y', 0.03887],
    ['60Y', 0.03772]],
    columns=['axis1', 'values']).set_index('axis1')
data_swap = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap)

In [5]:
### ois basis swap
df_basis_swap = pd.DataFrame([    
    ['1Y', 0.0005],
    ['2Y', 0.0003],
    ['3Y', 0.0003],
    ['4Y', 0.0002],
    ['5Y', 0.0001]],
    columns=['axis1', 'values']).set_index('axis1')
data_basis_swap = qfCreateData1D('OVERNIGHT INDEX BASIS SWAP', 'USD-FF-3M-OVER-USD-SOFR-OIS-3M', df_basis_swap)

In [6]:
### spread zero
df_spread_zero_rate_rfr = pd.DataFrame(
    [['1Y', 0.],
     ['5Y', 0.],
     ['10Y', 0.],
     ['20Y', 0.],
     ['30Y', 0.]], 
    columns=['axis1', 'values']).set_index('axis1')
data_szr_rfr = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate_rfr)
df_spread_zero_rate_ff = pd.DataFrame(
    [['1Y', 0.0002],
     ['5Y', 0.0003],
     ['10Y', 0.0003],
     ['20Y', 0.0003],
     ['30Y', 0.0004]], 
    columns=['axis1', 'values']).set_index('axis1')
data_szr_ff = qfCreateData1D('SPREAD ZERO RATE', 'FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD', df_spread_zero_rate_ff)

In [7]:
### funding parameter table
df_dg = pd.DataFrame([
    ['Overnight Index Future', 'SOFR-FUTURE-3M', 'SOFR-1B-FLAT'],
    ['Overnight Index Swap', 'USD-SOFR-OIS', 'SOFR-1B-FLAT'],
    ['Overnight Index Basis Swap', 'USD-FF-3M-OVER-USD-SOFR-OIS-3M', 'SOFR-1B-FLAT']], 
    columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_dg)

In [8]:
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_fut, data_swap, data_basis_swap, data_szr_rfr, data_szr_ff, data_fpt])

### Create Model Yield Curve

In [9]:
value_date = '2026-02-11'
yc_model = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)
path = 'serialized/yc_model_multi_indices_calibrated.pickle'
qfWriteModelObjectToFile(yc_model, path)

'DONE'

### Re-Price Calibration Instruments

In [10]:
### everything is collateralised in RFR
fi_vp = qfCreateValuationParameters('FUNDING INDEX PARAMETER', {'Funding Index' : 'SOFR-1B-FLAT'})
vp_collection = qfCreateValuationParametersCollection([fi_vp])
### re-price each product
req = 'parrateorspread'
dc_display = data_collection.display()
for data in data_collection:
    if data.data_shape != 'DATAGENERIC' and data.data_type != 'SPREAD ZERO RATE':
        for _, row in data.display().iterrows():
            axis, quote = row['axis1'], row['values']
            this_product = qfCreateProductFromDataConvention(value_date, data.data_convention.name, axis, quote)
            par_rate = qfCreateValueReport(yc_model, this_product, vp_collection, req)
            if 'FUTURE' in data.data_type:
                print(f'target is {quote:.2f}, and model implied is {100 - 100 * par_rate:.2f}.')
            else:
                print(f'target is {quote:.2%}, and model implied is {par_rate:.2%}.')
                pv = qfCreateValueReport(yc_model, this_product, vp_collection, 'pv')
                print(f'Pv of the swap is {pv[0][1]:.2f}.')


target is 96.44, and model implied is 96.44.
target is 96.70, and model implied is 96.70.
target is 96.85, and model implied is 96.85.
target is 96.90, and model implied is 96.90.
target is 96.91, and model implied is 96.91.
target is 96.89, and model implied is 96.89.
target is 96.85, and model implied is 96.85.
target is 96.81, and model implied is 96.81.
target is 96.76, and model implied is 96.76.
target is 96.71, and model implied is 96.71.
target is 96.66, and model implied is 96.66.
target is 96.61, and model implied is 96.61.
target is 3.36%, and model implied is 3.36%.
Pv of the swap is -0.00.
target is 3.42%, and model implied is 3.42%.
Pv of the swap is -0.00.
target is 3.49%, and model implied is 3.49%.
Pv of the swap is -0.00.
target is 3.56%, and model implied is 3.56%.
Pv of the swap is -0.00.
target is 3.62%, and model implied is 3.62%.
Pv of the swap is -0.00.
target is 3.69%, and model implied is 3.69%.
Pv of the swap is -0.00.
target is 3.74%, and model implied is 3.

### Discount Factor Check

In [13]:
ff_fi = 'FF-1B' # be cautious, this is not a pure disc factor it is implied from the rfr disc ff fwd rate
rfr_fi = 'SOFR-1B'
expiry_date = '2027-10-19'
expiry_date_adj = qfMoveToBusinessDay(expiry_date, 'F', 'USGS')
df_ff = qfDiscountFactor(yc_model, ff_fi, expiry_date_adj)
df_rfr = qfDiscountFactor(yc_model, rfr_fi, expiry_date_adj)
print(f'Discount factor up to {expiry_date_adj} for ff is: {df_ff:.4f}, for rfr is {df_rfr:.4f}.')

Discount factor up to 2027-10-19 for ff is: 0.9458, for rfr is 0.9465.


### Analytic Index Value Check

In [12]:
### calculate FF-1B term rate
index_ff = 'FF-1B'
index_rfr = 'SOFR-1B'
effective_date = '2026-10-19'
term_or_termination_date = '2027-01-19'
fwd_ff = qfValueIndexForward(yc_model, vp_collection, index_ff, effective_date, term_or_termination_date)
print(f'Forward level for {index_ff} is {fwd_ff:.2%}.')
fwd_rfr = qfValueIndexForward(yc_model, vp_collection, index_rfr, effective_date, term_or_termination_date)
print(f'Forward level for {index_rfr} is {fwd_rfr:.2%}.')
# makes sense, sofr < ff

Forward level for FF-1B is 3.37%.
Forward level for SOFR-1B is 3.13%.
